In [ ]:
import math

class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data              # number
        self.grad = 0.0               # dL/d(this), modatlo 0
        self._backward = lambda: None # gradient venakki pampe rule (tarvata set chestham)
        self._prev = set(_children)   # ye numbers nunchi puttindi
        self._op = _op                # ye operation tho ('+', '*', 'tanh')
        self.label = label            # peru (debug kosam)

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

In [ ]:
def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
        self.grad  += 1.0 * out.grad
        other.grad += 1.0 * out.grad
    out._backward = _backward
    return out

def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    def _backward():
        self.grad  += other.data * out.grad
        other.grad += self.data  * out.grad
    out._backward = _backward
    return out

def __radd__(self, other): return self + other   # 1 + Value
def __rmul__(self, other): return self * other   # 2 * Value

Value.__add__ = __add__
Value.__mul__ = __mul__
Value.__radd__ = __radd__
Value.__rmul__ = __rmul__

In [ ]:
def tanh(self):
    t = math.tanh(self.data)
    out = Value(t, (self,), 'tanh')
    def _backward():
        self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    return out

Value.tanh = tanh

In [ ]:
def backward(self):
    # 1. Topological sort: prathi node ni, daani parents tarvata order lo pettu
    topo, visited = [], set()
    def build(v):
        if v not in visited:
            visited.add(v)
            for child in v._prev:
                build(child)
            topo.append(v)
    build(self)

    # 2. Final output ki grad = 1 (dL/dL = 1)
    self.grad = 1.0

    # 3. Reverse order lo prathi node _backward call
    for v in reversed(topo):
        v._backward()

Value.backward = backward

In [ ]:
a = Value(2.0); b = Value(-3.0); c = Value(10.0); f = Value(-2.0)
e = a * b      # -6
d = e + c      #  4
L = d * f      # -8

L.backward()
for name, v in [("L", L), ("d", d), ("f", f), ("e", e), ("c", c), ("a", a), ("b", b)]:
    print(name, v)

L Value(data=-8.0000, grad=1.0000)
d Value(data=4.0000, grad=-2.0000)
f Value(data=-2.0000, grad=4.0000)
e Value(data=-6.0000, grad=-2.0000)
c Value(data=10.0000, grad=-2.0000)
a Value(data=2.0000, grad=6.0000)
b Value(data=-3.0000, grad=-4.0000)


In [ ]:
x1 = Value(2.0);  x2 = Value(0.0)      # inputs
w1 = Value(-3.0); w2 = Value(1.0)      # weights
b  = Value(6.8813735870195432)         # bias

n = x1*w1 + x2*w2 + b                  # ninnati  X @ W + b, okka neuron ki
o = n.tanh()                           # activation
o.backward()

for name, v in [("o", o), ("x1", x1), ("w1", w1), ("x2", x2), ("w2", w2), ("b", b)]:
    print(name, v)

o Value(data=0.7071, grad=1.0000)
x1 Value(data=2.0000, grad=-1.5000)
w1 Value(data=-3.0000, grad=1.0000)
x2 Value(data=0.0000, grad=0.5000)
w2 Value(data=1.0000, grad=0.0000)
b Value(data=6.8814, grad=0.5000)


In [7]:
a = Value(3.0)
b = a + a          # b = 6
b.backward()
print(a)           # grad = 2.0 undali (d(a+a)/da = 2)

Value(data=3.0000, grad=2.0000)


In [8]:
import torch

x1 = torch.tensor(2.0, dtype=torch.double, requires_grad=True)
x2 = torch.tensor(0.0, dtype=torch.double, requires_grad=True)
w1 = torch.tensor(-3.0, dtype=torch.double, requires_grad=True)
w2 = torch.tensor(1.0, dtype=torch.double, requires_grad=True)
b  = torch.tensor(6.8813735870195432, dtype=torch.double, requires_grad=True)

o = torch.tanh(x1*w1 + x2*w2 + b)
o.backward()

print("PyTorch:", o.item(), x1.grad.item(), w1.grad.item(), x2.grad.item(), w2.grad.item(), b.grad.item())

PyTorch: 0.7071067811865476 -1.4999999999999998 0.9999999999999999 0.49999999999999994 0.0 0.49999999999999994
